# Inference using TRVI

This notebook uses the trained TRVI model to infer relevance-weighted cell embeddings of gene expression and alternative splicing-induced transcript usage. The inferred cell embeddings are then used for Crecerelle's downstream workflows such as

*   UMAP visualisation
*   Integration of gene expression and transript usage
*   Modality-relevance weight analysis
*   differential gene expression analysis
*   differential splicing analysis
*   data-driven cell annotation
*   cell cluster-specific marker gene determination
*   cell cluster-specific isoform determination
*   analysis of cluster-specific isoform regulations
*   functional enrichment analysis
*   biological pathway analysis associated with differentially spliced and expressed genes


This notebooks requires to have the trained models and AnnData objects uploaded in the file directory system as described in Set-up. Start with running Set-up and continue with the subsequent steps.

# Set-up

In [ ]:
import os

# @markdown **Crecerelle:** Install the library from GitHub
!pip install git+https://github.com/Hollfelder-Lab/crecerelle.git

In [ ]:
# @markdown **Directory structure:** Set-up the directory structure to store the results of Crecerelle. We recommend connecting to your Google Drive. Otherwise, the directories will be created only temporarily in Colab. If you have already set-up the directory structure from a previous analysis, the existing directories will not be overwritten but only the new ones added.

connect_google_drive = True # @param {type:"boolean"}

if connect_google_drive:
    from google.colab import drive
    drive.mount('/content/drive')

    root_directory = "/content/drive/My Drive"
else:
    root_directory = "/content"





The following directory structure is then created:
```text
root_directory/
└── crecerelle_results/
    ├── data/
    │   └── <dataset_name>/  *(optional)*
    ├── figures/
    │   └── <dataset_name>/  *(optional)*
    └── models/
        ├── TRVI/
        ├── tuVI/
        └── scVI/

In [ ]:
from crecerelle.utils import setup_crecerelle

# @markdown **Dataset name:** Define the dataset name
dataset_name = "tabulaMuris" # @param

# Set-up the directory structure
setup_crecerelle(root_directory, dataset_name=dataset_name)

# @markdown The working directory is then set to "./crecerelle_results"
if os.path.basename(os.path.normpath(os.getcwd())) != "crecerelle_results":
    os.chdir("drive/My Drive/crecerelle_results")

# Print the working directory
print(f"The working directory is set to: {os.getcwd()}")

In [ ]:
# @markdown **Imports:** modules from PyTorch, Numpy, Scanpy amongst others are imported together with the crecerelle modules necessary for TRVI
# General import
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
from umap import UMAP
import scib
import pdb
import pandas as pd

# Imports from crecerelle
from crecerelle.models import TRVI
from crecerelle.cell import TABULA_MURIS_CELL_TYPES, MOUSE_CORTEX_BICCN_CELL_TYPES, TABULA_MURIS_TISSUE_ORGAN_SYSTEM_DICT, TABULA_MURIS_CELL_TYPE_ABBREVIATION_DICT
from crecerelle.utils import GeneExpressionTranscriptUsageDataset, intron_names_2_integers

# Settings for plotting
plt.rcParams.update({
    'font.size': 16,              # General font size
    'axes.labelsize': 16,         # Font size for x and y labels
    'axes.titlesize': 16,         # Font size for subplot titles
    'xtick.labelsize': 16,        # Font size for x-axis tick labels
    'ytick.labelsize': 16,        # Font size for y-axis tick labels
    'legend.fontsize': 16,        # Font size for legend labels
    #'font.family': 'sans-serif',
    #'font.sans-serif': 'Roboto',
    # 'figure.titlesize': 10,    # Font size for the overall figure title (if you use it)
})

In [ ]:
# Run for work
#import os
#os.chdir("../scASpred")

#!pwd

In [ ]:
# @markdown **Specify settings:** These are the general settings for keys to be defined. They are stored in a dictionary called settings. The dataset_name key is already saved in it. The current settings contain the default version for the Tabula Muris dataset.

# @markdown Key of Anndata object specifiying the cell type
settings_cell_type_key = "cell_ontology_class" # @param

# @markdown  Cell types of dataset given
settings_cell_types = TABULA_MURIS_CELL_TYPES # @param

# @markdown Abbreviations of cell types
settings_cell_type_abbreviations = TABULA_MURIS_CELL_TYPE_ABBREVIATION_DICT # @param

# @markdown Organisational entity of cell groups (e.g. tissue)
settings_cell_type_groups_key = "tissue" # @param

# @markdown Name of gene annotation file
settings_gene_annotation = "Mus_musculus.GRCm38.102.chr.gtf.gz" # @param

# @markdown Number of highly variable genes selected (HVGs)
settings_num_hvg = 3000 # @param

# @markdown Data partition used (recommended: "full") for tasks
settings_data_type = "full" # @param

# @markdown Key of Anndata object specifying the data partition (e.g. "train", "val", "test)
settings_data_partition_key = "data_partition" # @param

# @markdown Filter data to cell types with a minimum count
settings_filter_cell_types = True # @param {type:"boolean"}

# @markdown Minimum count threshold for filtering if filter_cell_types is True
settings_min_count_threshold = 100 # @param {type:"integer"}

settings = {
    "dataset_name": dataset_name, # Name of dataset
    "cell_type_key": settings_cell_type_key, # Key of Anndata object specifiying the cell type
    "cell_types": settings_cell_types, # Cell types of dataset given
    "cell_type_abbreviations": settings_cell_type_abbreviations, # Abbreviations of cell types
    "cell_type_groups_key": settings_cell_type_groups_key, # Organisational entity of cell groups (e.g. tissue)
    "gene_annotation": settings_gene_annotation,
    "num_hvg": settings_num_hvg, # Number of highly variable genes selected (HVGs)
    "data_type": settings_data_type, # Data partition used (recommended: "full") for tasks
    "data_partition_key": settings_data_partition_key,
    "filter_cell_types": settings_filter_cell_types, # Filter data to cell types with a minimum count
    "min_count_threshold": settings_min_count_threshold, # Minimum count threshold for filtering if filter_cell_types is True
}

In [ ]:
# @markdown All data files must be stored here and the resulting new data will also be stored here.
path2data = "./data/" + settings["dataset_name"] + "/"

print(f"The data files will be stored in: {path2data}")

# Loading of data




Ensure the Anndata objects of the gene expression and transcript usage data are uploaded in "./crecerelle_results/data/dataset_name" *(optional)*. The object containing the gene expression matrix should have the prefix "adata_GE" whereas the one containing the transcript usage matrix should start with "adata_TU". Examples are:

*   adata_GE_{data_type}_preprocessed.h5ad
    *   e.g. adata_GE_full_preprocessed.h5ad
*   adata_TU_{data_type}_preprocessed.h5ad
    *   e.g. adata_TU_full_preprocessed.h5ad
*   adata_GE_{num_hvg}_{data_partition}.h5ad
    *   adata_GE_3000_train.h5ad
    *   adata_GE_3000_val.h5ad
    *   adata_GE_3000_test.h5ad

## Gene expression and transcript usage

In [ ]:
# @markdown **Dataset:** Choose the dataset for inference with TRVI

hvg_annotation = np.load(path2data + "full_" + str(settings["num_hvg"]) + "_hvg_annotation.npy")

if settings["data_type"]  != "full":
    # Load gene expression data of specified data partition
    adata_GE = ad.read_h5ad(path2data + "adata_GE_" + settings["data_type"]  + "_preprocessed.h5ad")
    adata_GE = adata_GE[:, hvg_annotation].copy()
    adata_GE.obs[settings["data_partition_key"]] = settings["data_type"]

    print(f"Following Anndata object contains the gene expression data of the data partition {settings["data_type"]}: \n {adata_GE}")

    # Load transcript usage data of specified data partition
    adata_TU = ad.read_h5ad(path2data + "adata_TU_" + settings["data_type"]  + "_preprocessed.h5ad")
    intron_groups_by_name = adata_TU.var["intron_group"].values
    intron_groups = intron_names_2_integers(intron_groups_by_name)
    unique_intron_groups, _ = np.unique(intron_groups, return_index=True)
    num_intron_groups = len(unique_intron_groups)
    adata_TU.obs[settings["data_partition_key"]] = settings["data_type"]

    print(f"Following Anndata object contains the transcript usage data of the data partition {settings["data_type"]}: \n {adata_TU}")
else:
    # Load gene expression training data
    adata_GE_train = ad.read_h5ad(path2data + "adata_GE_" + str(settings["num_hvg"]) + "_train.h5ad")
    adata_GE_train = adata_GE_train[:, hvg_annotation].copy()
    adata_GE_train.obs[settings["data_partition_key"]] = "train"

    # Load transcript usage training data
    adata_TU_train = ad.read_h5ad(path2data + "adata_TU_" + str(settings["num_hvg"]) + "_train.h5ad")
    intron_groups_by_name = adata_TU_train.var["intron_group"].values
    intron_groups = intron_names_2_integers(intron_groups_by_name)
    unique_intron_groups, _ = np.unique(intron_groups, return_index=True)
    num_intron_groups = len(unique_intron_groups)
    adata_TU_train.obs[settings["data_partition_key"]] = "train"

    # Load gene expression validation data
    adata_GE_val = ad.read_h5ad(path2data + "adata_GE_" + str(settings["num_hvg"]) + "_val.h5ad")
    adata_GE_val = adata_GE_val[:, hvg_annotation].copy()
    adata_GE_val.obs[settings["data_partition_key"]] = "val"

    # Load transcript usage validation data
    adata_TU_val = ad.read_h5ad(path2data + "adata_TU_" + str(settings["num_hvg"]) + "_val.h5ad")
    adata_TU_val.obs[settings["data_partition_key"]] = "val"

    # Load gene expression test data
    adata_GE_test = ad.read_h5ad(path2data + "adata_GE_" + str(settings["num_hvg"]) + "_test.h5ad")
    adata_GE_test = adata_GE_test[:, hvg_annotation].copy()
    adata_GE_test.obs[settings["data_partition_key"]] = "test"

    # Load transcript usage test data
    adata_TU_test = ad.read_h5ad(path2data + "adata_TU_" + str(settings["num_hvg"]) + "_test.h5ad")
    adata_TU_test.obs[settings["data_partition_key"]] = "test"

    if settings["filter_cell_types"]:
        cell_types, cell_counts = np.unique(adata_GE_train.obs[settings["cell_type_key"]].to_numpy(), return_counts=True)
        filtered_cell_types = cell_types[np.argwhere(cell_counts >= settings["min_count_threshold"])].squeeze()

        # Filter adata_GE_train, adata_GE_val, adata_GE_test to filtered_cell_types
        adata_GE_train_filtered = adata_GE_train[adata_GE_train.obs[settings["cell_type_key"]].isin(filtered_cell_types)].copy()
        adata_GE_val_filtered = adata_GE_val[adata_GE_val.obs[settings["cell_type_key"]].isin(filtered_cell_types)].copy()
        adata_GE_test_filtered = adata_GE_test[adata_GE_test.obs[settings["cell_type_key"]].isin(filtered_cell_types)].copy()

        adata_GE_train = adata_GE_train_filtered
        adata_GE_val = adata_GE_val_filtered
        adata_GE_test = adata_GE_test_filtered

        # Filter adata_TU_train, adata_TU_val, adata_TU_test accordingly
        adata_TU_train_filtered = adata_TU_train[adata_TU_train.obs[settings["cell_type_key"]].isin(filtered_cell_types)].copy()
        adata_TU_val_filtered = adata_TU_val[adata_TU_val.obs[settings["cell_type_key"]].isin(filtered_cell_types)].copy()
        adata_TU_test_filtered = adata_TU_test[adata_TU_test.obs[settings["cell_type_key"]].isin(filtered_cell_types)].copy()

        adata_TU_train = adata_TU_train_filtered
        adata_TU_val = adata_TU_val_filtered
        adata_TU_test = adata_TU_test_filtered

        print(f"Cell types with less than {settings["min_count_threshold"]} have been filtered out. \n")

    adata_GE = ad.concat([adata_GE_train, adata_GE_val, adata_GE_test], axis=0, join="outer", merge="same")
    adata_TU = ad.concat([adata_TU_train, adata_TU_val, adata_TU_test], axis=0, join="outer", merge="same")

# @markdown **Optional: Filter cells in taxonomy level:** Per taxonomy level, filter out all cell types with less than min_cells (if not set to 0)
min_cells_per_tax_level = 50 # @param {type:"integer"}

if min_cells_per_tax_level > 0:
    # 1. Calculate the size of each cell type group within each tissue
    # transform('size') returns a Series with the same index as the original dataframe
    counts = adata_TU.obs.groupby([settings["cell_type_groups_key"], settings["cell_type_key"]])[settings["cell_type_key"]].transform('size')

    # 2. Create a boolean mask for cells to keep
    keep_mask = counts >= min_cells_per_tax_level

    # 3. Slice both AnnData objects in the tuple
    adata_TU_filtered = adata_TU[keep_mask, :].copy()
    adata_GE_filtered = adata_GE[keep_mask, :].copy()

    adata_TU_filtered.obs[settings["cell_type_key"]] = adata_TU_filtered.obs[settings["cell_type_key"]].cat.remove_unused_categories()
    adata_GE_filtered.obs[settings["cell_type_key"]] = adata_GE_filtered.obs[settings["cell_type_key"]].cat.remove_unused_categories()

    adata_TU = adata_TU_filtered.copy()
    adata_TU_train = adata_TU[adata_TU.obs[settings["data_partition_key"]] == "train", :].copy()
    adata_TU_val = adata_TU[adata_TU.obs[settings["data_partition_key"]] == "val", :].copy()
    adata_TU_test = adata_TU[adata_TU.obs[settings["data_partition_key"]] == "test", :].copy()

    adata_GE = adata_GE_filtered.copy()
    adata_GE_train = adata_GE[adata_GE.obs[settings["data_partition_key"]] == "train", :].copy()
    adata_GE_val = adata_GE[adata_GE.obs[settings["data_partition_key"]] == "val", :].copy()
    adata_GE_test = adata_GE[adata_GE.obs[settings["data_partition_key"]] == "test", :].copy()

    print(f"Cell types with less than {min_cells_per_tax_level} cells per taxonomy level have been filtered out. \n")

print(f"Following Anndata object contains the gene expression data of the data partition {settings["data_type"]}: \n {adata_GE}")
print(f"Following Anndata object contains the transcript usage data of the data partition {settings["data_type"]}: \n {adata_TU}")


### Create dataset

In [ ]:
# Extract gene expression data from AnnData object
cell_gene_counts = torch.from_numpy(adata_GE.layers["counts"].toarray())
cell_gene_levels = torch.from_numpy(adata_GE.X.toarray())

transform_gene_counts = None
transform_ontology = "one-hot"

# Extract transript usage training data from AnnData object
cell_intron_counts = torch.from_numpy(adata_TU.layers["counts"].toarray())
#cell_intron_levels = torch.from_numpy(adata_TU.layers["psi"])
cell_intron_levels = torch.from_numpy(adata_TU.X.toarray())
transform_intron_counts = None # Previously "log" but incorrect
transform_intron_levels = None

# @markdown **Dataset:** Create the combined dataset of gene expression and transcript usage for inference with TRVI
gene_transcript_dataset = GeneExpressionTranscriptUsageDataset(
    cell_gene_counts=cell_gene_counts,
    cell_gene_levels=cell_gene_levels,
    cell_intron_counts=cell_intron_counts,
    cell_intron_levels=cell_intron_levels,
    ontology=adata_GE.obs[settings["cell_type_key"]].to_numpy(),
    tissue=adata_GE.obs[settings["cell_type_groups_key"]].to_numpy(),
    cell_types=cell_types,
    transform_gene_counts=transform_gene_counts,
    transform_intron_counts=transform_intron_counts,
    transform_intron_levels=transform_intron_levels,
    transform_ontology=transform_ontology
)

print(f"The dataset is set up for inference.")

# Model

In [ ]:
# @markdown **Dimension of latent spaces:** Define the dimensionality of the private gene expression embeddings (latent_dim_1), of the private transcript usage embeddings (latent_dim_2), and the shared embeddings (latent_dim_shared)

# input_dim[0] for gene expression, input_dim[1] for transcript usage
input_dim = [gene_transcript_dataset.num_genes, gene_transcript_dataset.num_introns]

# number of intron groups for transcript usage # TO DO ELIMINATE THIS PARAMETER LATER
num_intron_groups = num_intron_groups

# intron groups for transcript usage # TO DO ELIMINATE THIS PARAMETER LATER
intron_groups = intron_groups

# latent_dim[0] for gene expression, latent_dim[1] for transcript usage, latent_dim[2] for shared latent
latent_dim_1 = 10 # @param {type:"integer"}
latent_dim_2 = 10 # @param {type:"integer"}
latent_dim_shared = 10 # @param {type:"integer"}
latent_dim = [latent_dim_1, latent_dim_2, latent_dim_shared]

# @markdown **Beta parameter:** Choose the value for the beta parameter for gene expression and transcript usage
# beta[0] for gene expression, beta[1] for transcript usage
beta_1 = 1.0 # @param {type:"number"}
beta_2 = 1.0 # @param {type:"number"}
beta = [beta_1, beta_2]

# @markdown **Architecture of neural networks:** choose the number of hidden layers as well as the number of hidden units per layer for the encoding and decoding neural networks of gene expression (num_hidden_layers_1, num_hidden_units_1) and transcript usage (num_hidden_layers_2, num_hidden_units_2)
# num_hidden_layers[0] for gene expression, num_hidden_layers[1] for transcript usage
num_hidden_layers_1 = 2 # @param {type:"integer"}
num_hidden_layers_2 = 2 # @param {type:"integer"}
num_hidden_layers = [num_hidden_layers_1, num_hidden_layers_2]

# num_hidden_units[0] for gene expression, num_hidden_units[1] for transcript usage
num_hidden_units_1 = 256 # @param {type:"integer"}
num_hidden_units_2 = 256 # @param {type:"integer"}
num_hidden_units = [num_hidden_units_1, num_hidden_units_2]

# @markdown **Observation model:** Choose the data likelihood of the gene expression data (likelihood_1) and of the transcript usage data (likelihood_2).
# likelihoods[0] for gene expression, likelihoods[1] for transcript usage
likelihood_1 = "ZINB" # @param ["ZINB", "NB","Gaussian"]
likelihood_2 = "ZIDM" # @param ["ZANIDM", "ZIDM", "DM"]
likelihoods = [likelihood_1, likelihood_2]

# @markdown **Data modality weighting:** Tick the box, if the importance of each modality for each data point shall be learnt
# Learn weighting of modalities
learn_modality_weighting = True # @param {type:"boolean"}

# @markdown **Dropout rate:** If dropout shall be applied, set the dropout rate to 0.0 > p > 1.0. Otherwise, choose 0.0.
dropout_rate = 0.1 # @param {type:"number"}
device = "cuda" # @param ["cuda", "cpu"]

scaling_factor_1 = False # @param {type:"boolean"}
scaling_factor_2 = False # @param {type:"boolean"}
scaling_factor = [scaling_factor_1, scaling_factor_2]

# Convert beta to string
for beta_i in beta:
    beta_str = f"{beta_i:.1f}"
    beta_str = beta_str.replace(".", "")
    while len(beta_str) < 3:
        beta_str = "0" + beta_str
    beta_str = beta_str

loss_type = "VariationalELBO"

## TRVI with ZINB and ZIDM

In [ ]:
# @markdown **Path to models:** Ensure all scTUVI models trained have been saved in this path
settings_path2models = "./models/TRVI/" # @param
settings["path2models"] = settings_path2models

print(f"The TRVI models trained are chosen from the path {settings["path2models"]}")

In [ ]:
# @markdown **Observation model for gene expression:** Zero-inflated negative binomial likelihood
likelihood_1 = "ZINB"
# @markdown **Observation model for transript usage:** Zero-inflated Dirichlet-Multinomial likelihood
likelihood_2 = "ZIDM"
# likelihoods[0] for gene expression, likelihoods[1] for transcript usage
likelihoods = [likelihood_1, likelihood_2]

model = TRVI(
    input_dim=input_dim,
    num_intron_groups=num_intron_groups,
    intron_groups=intron_groups,
    latent_dim=latent_dim,
    beta=beta,
    num_hidden_layers=num_hidden_layers,
    num_hidden_units=num_hidden_units,
    likelihoods=likelihoods,
    scaling_factor=scaling_factor,
    temp=0.5,
    learn_modality_weighting=learn_modality_weighting,
    dropout_rate=dropout_rate,
    device=device,
)

# @markdown **Random seed:** select the random seed
seed = 8 # @param {type:"integer"} 0, 1, 2, 3, 4, 5, 6, 7, 8, 9
torch.manual_seed(seed)
np.random.seed(seed)

model_name = "TRVI_"+ str(seed) + "_" + likelihoods[0] + "_" + likelihoods[1] + "_" + beta_str

# @markdown **Model checkpoint:** Specify the model checkpoint you want to load for inference
epoch_checkpoint = 324 # @param {type:"integer"}

model_checkpoint = model_name + "_epochs_" + str(epoch_checkpoint) + "_checkpoint.pth"
checkpoint = torch.load(settings["path2models"] + settings["dataset_name"] +"_" + str(settings["num_hvg"]) + "_" + model_checkpoint)
model.load_state_dict(checkpoint["model_state_dict"])

# Push model to GPU if available
if torch.cuda.is_available():
    model = model.to(device)

## TRVI with Gaussian and ZIDM

In [ ]:
# @markdown **Observation model for gene expression:** Gaussian likelihood
likelihood_1 = "Gaussian"
# @markdown **Observation model for transript usage:** Zero-inflated Dirichlet-Multinomial likelihood
likelihood_2 = "ZIDM"
# likelihoods[0] for gene expression, likelihoods[1] for transcript usage
likelihoods = [likelihood_1, likelihood_2]

# Inference with chosen model

In [ ]:
# @markdown **Inference with chosen TRVI:** The trained TRVI is used to infer the bi-modal cell embeddings of TRVI on the specified dataset
print(f"Following model is used for inference: \n {model}")

from crecerelle.utils import inference_trvi
adata_GE, adata_TU = inference_trvi(
    (adata_GE, adata_TU),
    gene_transcript_dataset,
    model,
    likelihoods,
    [True, True],
    256
)

print(f"Following Anndata object contains the gene expression data and inferences of the data partition {settings["data_type"]}: \n {adata_GE}")
print(f"Following Anndata object contains the transcript usage data and inferences of the data partition {settings["data_type"]}: \n {adata_TU}")

In [ ]:
# Random seed robustness
from crecerelle.utils import run_random_seed_evaluation_TRVI

seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
epoch_checkpoints = [603, 465, 550, 577, 413, 575, 310, 361, 324, 542]

batch_size = 256

# Run random seed evaluation
adata_GE, adata_TU = run_random_seed_evaluation_TRVI(
        model,
        likelihoods=likelihoods,
        beta_str=beta_str,
        list_of_random_seeds=seeds,
        list_of_epoch_checkpoints=epoch_checkpoints,
        adata=(adata_GE, adata_TU),
        dataset=gene_transcript_dataset,
        dataset_name=settings["dataset_name"],
        num_hvg=settings["num_hvg"],
        path2models=settings["path2models"],
        device=device,
        count_data_included=[True, True],
        batch_size=batch_size,
)

In [ ]:
adata_GE

In [ ]:
adata_TU

### Save AnnData objects with inferences

In [ ]:
# @markdown **Inferred bi-modal cell embeddings:** Tick the box if you want to save the inferred bi-modal cell embeddings with the data as a new AnnData object
save_inferred_data = True # @param {type:"boolean"}

if save_inferred_data:
    if settings["data_type"] != "full":
        adata_GE.write(path2data + "adata_GE_" + str(settings["num_hvg"]) + "_" + settings["data_type"] + "_trvi_inference.h5ad")
        adata_TU.write(path2data + "adata_TU_" + str(settings["num_hvg"]) + "_" + settings["data_type"] + "_trvi_inference.h5ad")
        #if tissue is None:
        #    cluster_eval_df.to_csv(path2data + "cluster_eval_" + data_type + "_" + likelihood_1 + "_" + likelihood_2 + "_trvi_inference.csv", index=False)
        #else:
        #    cluster_eval_df.to_csv(path2data + "cluster_eval_" + tissue + "_" + data_type + "_" + likelihood_1 + "_" + likelihood_2 + "_trvi_inference.csv", index=False)
    else:
        adata_GE.write(path2data + "adata_GE_" + str(settings["num_hvg"]) + "_" + "trvi_inference.h5ad")
        adata_TU.write(path2data + "adata_TU_" + str(settings["num_hvg"]) + "_" + "trvi_inference.h5ad")
        #if tissue is None:
        #    cluster_eval_df.to_csv(path2data + "cluster_eval_" + likelihood_1 + "_" + likelihood_2 + "_trvi_inference.csv", index=False)
        #else:
        #    cluster_eval_df.to_csv(path2data + "cluster_eval_" + tissue + "_" + likelihood_1 + "_" + likelihood_2 + "_trvi_inference.csv", index=False)

# Load inference results

Load the AnnData files containing the gene expression and transcript usage data and their embeddings learnt by TRVI

In [ ]:
# @markdown **Data partition:** Choose the data partition of which you want to load the inference results
data_partition = "full" # @param ["train", "val", "test", "train-test", "full"]

hvg_annotation = np.load(path2data + "full_" + str(settings["num_hvg"]) + "_hvg_annotation.npy")

# @markdown **Random seed:** select the random seed
seed = 8 # @param {type:"integer"} 0, 1, 2, 3, 4, 5, 6, 7, 8, 9
torch.manual_seed(seed)
np.random.seed(seed)

# Load full dataset
# Assert if files exists
path2GE_files = path2data + "adata_GE_" + str(settings["num_hvg"]) + "_trvi_inference.h5ad"
path2TU_files = path2data + "adata_TU_"+ str(settings["num_hvg"]) +"_trvi_inference.h5ad"
assert os.path.exists(path2GE_files), "File does not exist, run inference with TRVI to obtain results"
assert os.path.exists(path2TU_files), "File does not exist, run inference with TRVI to obtain results"

adata_GE = ad.read_h5ad(path2GE_files)
adata_TU = ad.read_h5ad(path2TU_files)

if data_partition == "train":
    adata_GE = adata_GE[adata_GE.obs[settings["data_partition_key"]] == "train"].copy()
    adata_TU = adata_TU[adata_TU.obs[settings["data_partition_key"]] == "train"].copy()
elif data_partition == "val":
    adata_GE = adata_GE[adata_GE.obs[settings["data_partition_key"]] == "val"].copy()
    adata_TU = adata_TU[adata_TU.obs[settings["data_partition_key"]] == "val"].copy()
elif data_partition == "test":
    adata_GE = adata_GE[adata_GE.obs[settings["data_partition_key"]] == "test"].copy()
    adata_TU = adata_TU[adata_TU.obs[settings["data_partition_key"]] == "test"].copy()
elif data_partition == "train-test":
    adata_GE = adata_GE[adata_GE.obs[settings["data_partition_key"]].isin(["train", "test"])].copy()
    adata_TU = adata_TU[adata_TU.obs[settings["data_partition_key"]].isin(["train", "test"])].copy()
elif data_partition == "full":
    pass
else:
    raise ValueError("Invalid data partition")

# Rename cells in .obs["cell_ontology_class"] according to cell type abbreviation dict
if settings["cell_type_abbreviations"] != None:
    adata_GE.obs[settings["cell_type_key"]] = adata_GE.obs[settings["cell_type_key"]].map(settings["cell_type_abbreviations"])
    adata_TU.obs[settings["cell_type_key"]] = adata_TU.obs[settings["cell_type_key"]].map(settings["cell_type_abbreviations"])

# Group into Organ systems: Map adata_GE.obs["tissue"] to organ systems according to organ system dictionary
if settings["dataset_name"] == "tabulaMuris":
    adata_GE.obs["organ_system"] = adata_GE.obs[settings["cell_type_groups_key"]].map(TABULA_MURIS_TISSUE_ORGAN_SYSTEM_DICT)
    adata_TU.obs["organ_system"] = adata_TU.obs[settings["cell_type_groups_key"]].map(TABULA_MURIS_TISSUE_ORGAN_SYSTEM_DICT)

# Create a tuple of gene expression and transcript usage AnnData objects
adata = (adata_GE, adata_TU)

print(f"Following inferred gene expression data have been loaded: \n {adata_GE} \n")
print(f"Following inferred transcript usage data have been loaded: \n {adata_TU}")

# Inference results benchmarking atlas level

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.image as mpimg
import seaborn as sns
from crecerelle.plotting_utils import plot_customized_UMAP_coordinates

# best_seed 8
# res GE 1.5
# res TU 1.8
# res GE-TU 1.5

# @markdown **Evaluation of TRVI cell embeddings:** Assess the learnt cell embeddings (S-GE, S-TU, Joint GE-TU) on the recovery of cell-type specific clustering"

# @markdown **Observation model of TRVI:** Select the gene expression and the transcript usage observation model that were used:
likelihood_GE = "ZINB" # @param ["ZINB", "NB","Gaussian"]
likelihood_TU = "ZIDM" # @param ["DM", "ZANIDM","ZIDM"]
likelihood_GE_TU = likelihood_GE + "_" + likelihood_TU

likelihood_keys = [likelihood_GE, likelihood_TU, likelihood_GE_TU]

# @markdown **Resolution for Leiden clustering:** Choose the resolution for the Leiden clustering for the cell embeddings (S-GE, S-TU, Joint GE-TU). The given values are the optimal ones on the Tabula Muris dataset
res_GE_shared = 1.5 # @param {type:"number"}
res_TU_shared = 1.8 # @param {type:"number"}
res_GE_TU_shared = 1.5 # @param {type:"number"}

resolutions = [res_GE_shared, res_TU_shared, res_GE_TU_shared]
nmi_list = []
ari_list = []
asw_list = []
avg_bio_list = []

for i, likelihood in enumerate(likelihood_keys):
    if likelihood == likelihood_GE:
        adata = adata_GE.copy()
    elif likelihood == likelihood_TU:
        adata = adata_TU.copy()
    elif likelihood == likelihood_GE_TU:
        adata = adata_GE.copy()
    else:
        raise ValueError("Invalid likelihood")


    # Calculate nearest neighbor graph
    embedding = likelihood + "_shared_latent_mean"
    sc.pp.neighbors(adata, use_rep=embedding)
    leiden_res = "leiden_res_" + str(resolutions[i])

    sc.tl.leiden(adata, key_added=leiden_res, resolution=resolutions[i], flavor="igraph")

    nmi_score = scib.me.nmi(adata, cluster_key=leiden_res, label_key=settings["cell_type_key"])
    asw_score = scib.me.silhouette(adata, label_key=settings["cell_type_key"],
                                          embed=embedding)
    ari_score = scib.me.ari(adata, cluster_key=leiden_res, label_key=settings["cell_type_key"])
    avg_bio_score = (nmi_score + asw_score + ari_score) / 3

    nmi_list.append(nmi_score)
    ari_list.append(ari_score)
    asw_list.append(asw_score)
    avg_bio_list.append(avg_bio_score)

# Create an evaluation dataframe
score_values = np.concatenate((np.array(nmi_list), np.array(ari_list), np.array(asw_list), np.array(avg_bio_list)))
if len(likelihood_keys) == 1:
    score_names = np.array(["NMI", "ARI", "ASW", "Avg Bio"])
else:
    score_nmi_names = ["NMI"] * len(likelihood_keys)
    score_ari_names = ["ARI"] * len(likelihood_keys)
    score_asw_names = ["ASW"] * len(likelihood_keys)
    score_avg_bio_names = ["Avg Bio"] * len(likelihood_keys)

    score_names = np.array(score_nmi_names + score_ari_names + score_asw_names + score_avg_bio_names)
likelihood_names = np.array((likelihood_keys * 4))

# In likelihood_names replace likelihood_GE with "S-GE", likelihood_TU with "S-TU", likelihood_GE_TU with "GE-TU"
likelihood_names = np.where(likelihood_names == likelihood_GE, "S-GE", likelihood_names)
likelihood_names = np.where(likelihood_names == likelihood_TU, "S-TU", likelihood_names)
likelihood_names = np.where(likelihood_names == likelihood_GE_TU, "GE-TU", likelihood_names)

evaluation_df = pd.DataFrame(
        {
            "Score value": score_values,
            "Evaluation score": score_names,
            "Embedding": likelihood_names
        }
)

atlas_mean_weight_GE = adata_GE.obs[likelihood_GE + "_weighting"].mean()
atlas_mean_weight_TU = adata_TU.obs[likelihood_TU + "_weighting"].mean()


print(f"The learnt cell embeddings have been evaluated on the task of clustering and cell-type recovery")

# @markdown **Save evaluation:** Save the evaluation of the TRVI embeddings as .csv file
save_evaluation = False # @param {type:"boolean"}

if save_evaluation:
    evaluation_df.to_csv(path2data + "evaluation_embeddings_trvi.csv", index=False)

evaluation_df




In [ ]:
# @markdown **Distribution of modality-relevance weights across cell types:** Across random seeds, evaluate the mean and standard deviation of the inferred relevance weight for gene expression or transcript usage for each cell type. Specify which transcriptomic facet of the modality-relevance weight (Gene expression or transcript usage) you want, and if you want to sort the cell types according to a key (e.g. "organ_system" in Tabula Muris, or None):

transcriptomic_facet = "Gene expression" # @param ["Gene expression", "Transcript usage"]

if transcriptomic_facet == "Gene expression":
    adata_selected = adata_GE
    likelihood_key = likelihood_GE
elif transcriptomic_facet == "Transcript usage":
    adata_selected = adata_TU
    likelihood_key = likelihood_TU
else:
    raise ValueError("Invalid transcriptomic facet")

sort_cell_type_key = "organ_system" # @param

# @markdown **Random seeds:** Specify the list of random seeds for evaluation (e.g. [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]). Note that there will be a check if the specified random seeds have been employed for inference. If not, carry out inference with the missing random seeds.

list_of_random_seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] # @param

from crecerelle.utils import analysis_trvi_relevance_weights_across_cell_types

mean_weights_df, atlas_weight_dict = analysis_trvi_relevance_weights_across_cell_types(
    adata_selected,
    list_of_random_seeds,
    transcriptomic_facet,
    likelihood_key,
    settings["cell_type_key"],
    sort_cell_type_key
)

likelihood_seeds_list = []

for random_seed in list_of_random_seeds:
    likelihood_seeds_list.append(likelihood_key+ "_" + str(random_seed))

mean_weights_transcriptomic_facet = mean_weights_df[likelihood_seeds_list].mean(axis=1)
std_weights_transcriptomic_facet = mean_weights_df[likelihood_seeds_list].std(axis=1)

mean_weights_df[likelihood_key + "_mean"] = mean_weights_transcriptomic_facet
mean_weights_df[likelihood_key + "_std"] = std_weights_transcriptomic_facet

print(f"The importance weights have been evaluated for {transcriptomic_facet}")
if transcriptomic_facet == "Gene expression":
    print(f"Overall the total mean modality-relevance weight for {transcriptomic_facet} is {atlas_weight_dict['total_mean_GE']}")
    print(f"Overall the total standard deviation of the modality-relevance weight for {transcriptomic_facet} is {atlas_weight_dict['total_std_GE']}")
if transcriptomic_facet == "Transcript usage":
    print(f"Overall the total mean modality-relevance weight for {transcriptomic_facet} is {atlas_weight_dict['total_mean_TU']}")
    print(f"Overall the total standard deviation of the modality-relevance weight for {transcriptomic_facet} is {atlas_weight_dict['total_std_TU']}")

mean_weights_df


In [ ]:
# @markdown **Save figure:**
save_fig = False # @param {type:"boolean"}
file_suffix = "pdf" # @param ["pdf", "png", "svg"]

# @markdown **UMAP colour display:** Choose the UMAP colour display (e.g. "organ_system", "cell_ontology_class")
umap_color_display_key = "organ_system" # @param

from crecerelle.plotting_utils import plot_latent_space_benchmarking_trvi
plot_latent_space_benchmarking_trvi(
        adata_objects=(adata_GE, adata_TU),
        likelihood_keys=[likelihood_GE, likelihood_TU],
        evaluation_df=evaluation_df,
        atlas_weight_dict=atlas_weight_dict,
        mean_weights_df=mean_weights_df,
        relevance_weight_likelihood_key=likelihood_key,
        seed=seed,
        random_seeds=list_of_random_seeds,
        umap_color_display_key=umap_color_display_key,
        sort_cell_type_key=sort_cell_type_key,
        save_fig=save_fig,
        dataset_name = settings["dataset_name"],
        file_suffix=file_suffix
)

Visualise the full latent space (P-GE, P-TU, S-GE, S-TU, GE-TU) of TRVI for the seed given (in paper we analyse the best performing seed (i.e. 8).

In [ ]:
from crecerelle.plotting_utils import plot_umap_latent_space_trvi

# @markdown **Display all latent spaces of TRVI:** The UMAPs of the latent spaces (P-GE, P-TU, S-GE, S-TU, GE-TU) are plotted colored according to the color keys provided (in Tabula Muris e.g. cell type, tissue, organ system).
color_keys = [settings["cell_type_key"], settings["cell_type_key"]] # @param
#color_keys = [settings["cell_type_groups_key"], settings["cell_type_groups_key"]]

# @markdown **Save figure:**
save_fig = False # @param {type:"boolean"}
file_suffix = "pdf" # @param ["pdf", "png", "svg"]


plot_umap_latent_space_trvi(
    adata_objects=(adata_GE, adata_TU),
    observation_model_keys=[likelihood_GE, likelihood_TU],
    color_keys=color_keys,
    seeds=[str(seed)],
    save_fig=save_fig,
    dataset_name=settings["dataset_name"],
    file_suffix=file_suffix,
    legend_title="Cell type",
    horizontal_legend_ncol=4,
    horizontal_legend_height_mm=52.0,
    horizontal_legend_gap_mm=5.0,
    legend_fontsize=6, # consider removing
    cell_type_legend_labelspacing=0.35,
    single_seed_figure_top=0.94,
    panel_label_y=1.02
)

In [ ]:
# Seed 8
# 1.5 for "ZINB_shared_latent_mean"
# 1.8 for "ZIDM_shared_latent_mean"
# 1.5 for "ZINB_ZIDM_shared_latent_mean"

In [ ]:
# Optimal leiden resolution for TRVI-ZINB
import scib
resolutions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9]
likelihood_key = "ZINB"
sc.pp.neighbors(adata_GE, use_rep="ZINB_8_shared_latent_mean")

res_opt, nmi_opt = scib.me.cluster_optimal_resolution(
            adata_GE,
            cluster_key="cluster",
            resolutions=resolutions,
            label_key=settings["cell_type_key"]
        )

In [ ]:
# Optimal leiden resolution for TRVI-ZIDM
import scib
resolutions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9]
sc.pp.neighbors(adata_TU, use_rep="ZIDM_8_shared_latent_mean")

res_opt, nmi_opt = scib.me.cluster_optimal_resolution(
            adata_TU,
            cluster_key="cluster",
            resolutions=resolutions,
            label_key=settings["cell_type_key"]
        )

# Inference results cluster differences on tissue and / or cell type level

In [ ]:
from crecerelle.plotting_utils import plot_cell_embeddings_umaps_trvi

# @markdown **Joint GE-TU embeddings:** Project the joint GE-TU embeddings as UMAPs for different taxonomy levels (e.g. different tissues) and (optionally) display their DEGs and DSGs

show_top_degs_dsgs = False # @param {type:"boolean"}

# @markdown **Taxonomy level list:** Name the taxonomy level list (e.g. different tissues) you want to visualise. Examples for Tabula Muris are (["Brain_Myeloid", "Brain_Non-Myeloid"], ["Aorta", "Heart", "Lung", "Trachea"], ["Diaphragm", "Limb_Muscle", "Tongue"], ["Large_Intestine", "Liver", "Pancreas"], ["BAT", "GAT", "MAT", "SCAT"], ["Marrow", "Spleen", "Thymus"], ["Bladder", "Mammary_Gland", "Skin", "Trachea"]). If you do not want to filter to a list of taxonomy levels choose None


#tax_level_list= ["Brain_Myeloid", "Brain_Non-Myeloid"] # Central Nervous System
#tax_level_list = ["Aorta", "Heart", "Lung", "Trachea"] # Cardiorespiratory System
#tax_level_list = ["Diaphragm", "Limb_Muscle", "Tongue"] # Musculoskeletal System
#tax_level_list = ["Large_Intestine", "Liver", "Pancreas"] # Digestive and Metabolic Organs
#tax_level_list = ["BAT", "GAT", "MAT", "SCAT"] # Adipose tissues
#tax_level_list = ["Marrow", "Spleen", "Thymus"] # Immune and hematopoietic System
#tax_level_list = ["Bladder", "Mammary_Gland", "Skin", "Trachea"] # Excretory and Integumentary organs
#tax_level_list = None

tax_level_list = None # @param


# @markdown **List of cells:** Name the cells e.g. you want to consider e.g. in Tabula Muris (["B cells", "T cells"]). If you do not want to filter. Choose None
# cell_list = ["B cell (pre)", "B cell (naive)", "B cell", "B cell (immature)", "B cell (late pro)"]
# cell_list = ["T cell", "T cell (CD4+)", "T cell (CD8+)", "DN4 thymocyte", "thymocyte"]
# cell_list = None
cell_list = ["T cell", "T cell (CD4+)", "T cell (CD8+)", "DN4 thymocyte", "thymocyte"] # @param

# @markdown **Save figure:** Tick box if you want to save the figure
save_fig = True # @param {type:"boolean"}

plot_cell_embeddings_umaps_trvi(
    adata_objects = (adata_GE, adata_TU),
    tissue_list = tax_level_list,
    cell_list = cell_list,
    likelihood_keys = ["ZINB", "ZIDM"],
    dataset_name = settings["dataset_name"],
    seed = seed,
    min_cells=50,
    save_fig = save_fig
)

# Inference results cell annotation, clustering, and disentanglement

The cell embeddings learnt by TRVI are used for cell annotation, differential gene expression and splicing analysis, and functional enrichment analysis.

## Differential gene expression analysis with TRVI

Run a differential gene expression analysis to find the top differentially expressed genes (DEGs) to annotate the Leiden clusters of the cell embeddings learnt by scGETUVI (i.e. S-GE, S-TU, joint GE-TU, P-GE, P-TU).

In [ ]:
# @markdown **Cell taxonomy level:** Select the cell taxonomy level (e.g. tissue, cell classes) to analyse e.g. (tissue: "Brain_Non-Myeloid", "Heart")
tax_level = "Brain_Non-Myeloid" # @param ["Aorta", "BAT","Bladder", "Brain_Myeloid","Brain_Non-Myeloid", "Diaphragm", "GAT", "Heart", "Large_Intestine", "Limb_Muscle", "Liver", "Lung", "Mammary_Gland","Marrow", "MAT", "Pancreas", "SCAT", "Skin", "Spleen", "Thymus", "Tongue", "Trachea"]
modality = "GE"

adata_GE_tissue = adata_GE[adata_GE.obs[settings["cell_type_groups_key"]] == tax_level].copy()

# @markdown **Filter cells on taxonomy level:** Filter out cell types with less than min_cells on taxonomy level

min_cells = 50 # @param {type:"integer"}

if min_cells > 0:
    cell_types, cell_counts = np.unique(adata_GE_tissue.obs[settings["cell_type_key"]].to_numpy(), return_counts=True)
    filtered_cell_types = cell_types[np.argwhere(cell_counts >= min_cells)].squeeze()

    # Ensure filtered_cell_types is a proper numpy array
    filtered_cell_types = np.atleast_1d(filtered_cell_types)

    # Filter adata_GE_train and adata_GE_val to filtered_cell_types
    adata_GE_tissue = adata_GE_tissue[adata_GE_tissue.obs[settings["cell_type_key"]].isin(filtered_cell_types)].copy()

# @markdown **Embedding:** Select the embedding of TRVI for differential gene expression analysis
embedding_key = "ZINB_ZIDM_shared" # @param ["ZINB_shared", "ZINB_ZIDM_shared", "ZINB_private"]

embedding = embedding_key + "_latent_mean"

model_name = "TRVI" + "_" + embedding_key

sc.pp.neighbors(adata_GE_tissue, use_rep=embedding, random_state=seed)

#adata_tissue.obsm["X_umap"] = adata_tissue.obsm[likelihood_key + "_X_umap"].copy()
adata_GE_tissue.obsm["X_umap"] = UMAP(n_components=2, random_state=seed).fit_transform(adata_GE_tissue.obsm[embedding])

if embedding_key == "ZINB_private":
    optimal_res = 0.20 # ZINB private [0.21, 0.29, 0.33]  0.21 is best resolution
    leiden_res = "leiden_res_" + str(optimal_res)
elif embedding_key == "ZINB_shared":
    optimal_res = 0.20 # ZINB shared [0.21, 0.29, 0.33]  0.21 is best resolution
    leiden_res = "leiden_res_" + str(optimal_res)
elif embedding_key == "ZINB_ZIDM_shared":
    optimal_res = 0.2 # ZINB ZIDM shared  (0.2 for Heart, 0.2 Brain Non-myeloid, Large Intestine), (Mammary Gland, GAT, Bladder),(0.4 Marrow, Limb Muscle, Skin, Pancreas),  (SCAT)
    leiden_res = "leiden_res_" + str(optimal_res)
else:
    raise ValueError("Invalid likelihood key")

# Clustering using Leiden algorithm
#sc.tl.leiden(adata_GE_tissue, key_added=leiden_res, resolution=optimal_res, flavor="igraph", random_state=seed)
sc.tl.leiden(adata_GE_tissue, resolution=optimal_res, flavor="igraph", random_state=seed)

# @markdown **Differential gene expression:** Select the cluster group for the analysis (e.g. Leiden, cell type key (i.e. cell_ontology_class in Tabula Muris))
#grouby = leiden_res
grouby = "leiden" # @param
# Obtain cluster-specific differentially expressed genes via the Wilcoxon test
sc.tl.rank_genes_groups(adata_GE_tissue, groupby=grouby, method="wilcoxon")

# @markdown **Save figure:**
save_fig = False # @param {type:"boolean"}

import matplotlib.gridspec as gridspec
from crecerelle.plotting_utils import plot_customized_UMAP_coordinates

# @markdown **Cluster group:** Select the cluster group of the clustering algorithm to consider
cluster_group = "0" # @param ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10"]

# Plot results
from crecerelle.plotting_utils import plot_deg_analysis

plot_deg_analysis(
    adata=adata_GE_tissue,
    cluster_group=cluster_group,
    model_name=model_name,
    taxonomy_level=tax_level,
    save_fig=save_fig,
    dataset_name=settings["dataset_name"],
    cell_type_key=settings["cell_type_key"]
)


In [ ]:
# @markdown **Differential gene expression:** Select the number of top differentially expressed genes to be displayed
num_deg = 200 # @param {type:"integer"}
sc.get.rank_genes_groups_df(adata_GE_tissue, group=cluster_group).head(num_deg)

In [ ]:
# @markdown **Threshold for pval_adj:** Select the threshold for significantly DEGs
sig_threshold = 0.05 # @param {type:"number"}

# Filter ranked gene groups to those smaller than sig_threshold
num_sig_degs = (sc.get.rank_genes_groups_df(adata_GE_tissue, group=cluster_group).head(num_deg)["pvals_adj"] < sig_threshold).to_numpy().sum()

print(f"Number of significantly DEGs in cluster {cluster_group}: {num_sig_degs}")

# @markdown Save adata_GE_tissue and rank_genes_groups_df as csv
save_adata_GE_tissue = False # @param {type:"boolean"}
if save_adata_GE_tissue:
    adata_GE_tissue.write(path2data + "adata_GE_" + str(settings["num_hvg"]) + "_" + tax_level + "_trvi_inference.h5ad")
    sig_rank_genes_groups_df = sc.get.rank_genes_groups_df(adata_GE_tissue, group=cluster_group).head(num_sig_degs)
    sig_rank_genes_groups_df.to_csv(path2data +"/deg_analysis_TRVI_" + embedding_key + "_" + tax_level + "_clustergroup_" + cluster_group + ".csv")
    print(f"The DEG analysis has been saved.")




In [ ]:
# @markdown Display the significant DEGs of the select cluster group
sig_deg_genes_group = sc.get.rank_genes_groups_df(adata_GE_tissue, group=cluster_group)["names"].to_numpy()[:num_sig_degs]
print(f"The significant DEGs of cluster group {cluster_group} are: \n {sig_deg_genes_group}")

## Differential splicing analysis with TRVI

Run a differential splicing analysis to find the top differentially spliced genes (DSGs) to annotate the Leiden clusters of of TRVI cell embeddings (i.e. S-GE, S-TU, joint GE-TU, P-GE, P-TU). We recommend joint GE-TU as cell embedding since it incorporates both the information of gene expression and transcript usage. We use the differential splicing test from scQuint.

In [ ]:
# @markdown Install the scQuint package for the differential splicing test
!pip install -U git+https://github.com/songlab-cal/scquint.git

In [ ]:
from scquint.data import add_gene_annotation, group_introns, filter_min_cells_per_feature, filter_min_cells_per_intron_group, calculate_PSI
from scquint.differential_splicing import run_differential_splicing, run_differential_splicing_for_each_group, find_marker_introns, mask_PSI
from scquint.dimensionality_reduction.pca import run_pca

# Make counts main layer again
if "counts" in adata_TU.layers:
    adata_TU.X = adata_TU.layers["counts"]
    adata_TU.layers.pop("counts")

# @markdown **Cell taxonomy level:** Select the cell taxonomy level (e.g. tissue, cell classes) to analyse e.g. (tissue: "Brain_Non-Myeloid", "Heart")
tax_level = "Brain_Non-Myeloid" # @param ["Aorta", "BAT","Bladder", "Brain_Myeloid","Brain_Non-Myeloid", "Diaphragm", "GAT", "Heart", "Large_Intestine", "Limb_Muscle", "Liver", "Lung", "Mammary_Gland","Marrow", "MAT", "Pancreas", "SCAT", "Skin", "Spleen", "Thymus", "Tongue", "Trachea"]
adata_TU_tissue = adata_TU[adata_TU.obs[settings["cell_type_groups_key"]] == tax_level].copy()

# @markdown **Filter cells on taxonomy level:** Filter out cell types with less than min_cells on taxonomy level

min_cells = 50 # @param {type:"integer"}

if min_cells > 0:
    cell_types, cell_counts = np.unique(adata_TU_tissue.obs[settings["cell_type_key"]].to_numpy(), return_counts=True)
    filtered_cell_types = cell_types[np.argwhere(cell_counts >= min_cells)].squeeze()

    # Ensure filtered_cell_types is a proper numpy array
    filtered_cell_types = np.atleast_1d(filtered_cell_types)

    # Filter adata_GE_train and adata_GE_val to filtered_cell_types
    adata_TU_tissue = adata_TU_tissue[adata_TU_tissue.obs[settings["cell_type_key"]].isin(filtered_cell_types)].copy()

# @markdown **Embedding:** Select the embedding of TRVI for differential splicing analysis
embedding_key = "ZINB_ZIDM_shared" # @param ["ZIDM_shared", "ZINB_ZIDM_shared", "ZIDM_private"]

embedding = embedding_key + "_latent_mean"

model_name = "TRVI" + "_" + embedding_key

if embedding_key == "ZIDM_shared":
    optimal_res = 0.2
    leiden_res = "leiden_res_" + str(optimal_res)
elif embedding_key == "ZINB_ZIDM_shared":
    optimal_res = 0.1 # (0.2 Heart, 0.2 Brain Non-myeloid, Large Intestine, 0.4 Marrow, Limb Muscle, Skin, Pancreas), 0.1 (Mammary Gland, GAT, Bladder), 0.3 (Limb Muscle, Skin, Pancreas), 0.2 (SCAT)
    leiden_res = "leiden_res_" + str(optimal_res)
elif embedding_key == "ZIDM_private":
    optimal_res = 0.2
    leiden_res = "leiden_res_" + str(optimal_res)
else:
    raise ValueError("Invalid embedding key")

# Compute UMAP on selected tissue
adata_TU_tissue.obsm[embedding_key + "_X_umap"] = UMAP(n_components=2, random_state=seed).fit_transform(adata_TU_tissue.obsm[embedding])

# Clustering using Leiden algorithm
sc.pp.neighbors(adata_TU_tissue, use_rep=embedding_key + "_latent_mean", random_state=seed)
#sc.tl.leiden(adata_TU_tissue, key_added=leiden_res, resolution=optimal_res, flavor="igraph", random_state=seed)
sc.tl.leiden(adata_TU_tissue, resolution=optimal_res, flavor="igraph", random_state=seed)


# Filter minmum number of cells per feature and intron group
adata_TU_tissue = filter_min_cells_per_feature(adata_TU_tissue, 100)
adata_TU_tissue = filter_min_cells_per_intron_group(adata_TU_tissue, 100)

# @markdown **Differential splicing analysis:** Select the cluster group for the analysis (e.g. Leiden, cell type key (i.e. cell_ontology_class in Tabula Muris))
dsg_group_key = "leiden" # @param

if dsg_group_key == "leiden":
    groups_test = adata_TU_tissue.obs[dsg_group_key].value_counts().index.tolist()

    if settings["dataset_name"] == "tabulaMuris" and tax_level == "Brain_Non-Myeloid":
        # Delete "5" from list
        groups_test = [x for x in groups_test if x != "5"]
    elif settings["dataset_name"] == "tabulaMuris" and tax_level == "Heart":
        # Delete "5" from list
        groups_test = [x for x in groups_test if x != "5"]
    elif settings["dataset_name"] == "tabulaMuris" and tax_level == "Large_Intestine":
        # Delete "5" from list
        groups_test = [x for x in groups_test if x != "5"]
    elif settings["dataset_name"] == "tabulaMuris" and tax_level == "Marrow":
        # Delete "7" from list
        groups_test = [x for x in groups_test if x != "7"]
    elif settings["dataset_name"] == "tabulaMuris" and tax_level == "GAT":
        # Delete "1" from list
        groups_test = [x for x in groups_test if x != "1"]
    elif settings["dataset_name"] == "tabulaMuris" and tax_level == "Skin":
        # Delete "7" from list
        groups_test = [x for x in groups_test if x != "2"]
    elif settings["dataset_name"] == "tabulaMuris" and tax_level == "SCAT":
        # Delete "7" from list
        groups_test = [x for x in groups_test if x not in ("4", "5")]


    """
    # Only for heart
    if tax_level == "Heart":
        groups_test = groups_test[:5]
    if tax_level == "Mammary_Gland":
        groups_test = groups_test[:3]
    if tax_level == "Large_Intestine":
        groups_test = groups_test[:4]
    """
elif dsg_group_key == "cell_ontology_class":
    groups_test = adata_TU_tissue.obs[dsg_group_key].value_counts().index.tolist()

    """
    if tax_level == "Limb_Muscle":
        groups_test = groups_test[:3]
    elif tax_level == "Diaphragm":
        groups_test = groups_test[:2]
    elif tax_level == "Spleen":
        groups_test = groups_test[:3]
    elif tax_level == "Trachea":
        groups_test = groups_test[:4]
    elif tax_level =="Lung":
        groups_test = groups_test[:2]
    elif tax_level == "SCAT":
        groups_test = groups_test[:4]
    """
else:
    raise ValueError("Invalid group key")

# Run differential splicing analysis
diff_spl_intron_groups, diff_spl_introns = run_differential_splicing_for_each_group(
    adata_TU_tissue, dsg_group_key, groups=groups_test, subset_to_groups=True,
    min_cells_per_intron_group=50, min_total_cells_per_intron=50,
    n_jobs=-1,  # -1 means use all cores
    # It will run much faster on a machine with multiple cores than in Colab
)





In [ ]:
groups_test

In [ ]:
# @markdown **Significance:** Determine the significantly spliced intron groups and select accordingly the significantly spliced introns
p_value_adj_thres = 0.05 # @param {type:"number"}
max_abs_delta_psi_thres = 0.05 # @param {type:"number"}

sig_diff_spl_intron_groups = diff_spl_intron_groups.query(f"p_value_adj < {p_value_adj_thres} and max_abs_delta_psi > {max_abs_delta_psi_thres}")
sig_spl_groups = list(sig_diff_spl_intron_groups["test_group"].unique())

#sig_diff_spl_introns = diff_spl_introns[diff_spl_introns["intron_group"].isin(sig_diff_spl_intron_groups["name"].to_list())]

sig_diff_spl_introns_df_list = []

for sig_group in sig_spl_groups:
    # Filter diff_spl_introns and sig_diff_spl_intron_groups to sig_group
    diff_spl_introns_sig_group = diff_spl_introns[diff_spl_introns["test_group"] == sig_group]
    sig_diff_spl_intron_groups_sig_group = sig_diff_spl_intron_groups[sig_diff_spl_intron_groups["test_group"] == sig_group]
    sig_diff_spl_introns_sig_group = diff_spl_introns_sig_group[diff_spl_introns_sig_group["intron_group"].isin(sig_diff_spl_intron_groups_sig_group["name"].to_list())]

    sig_diff_spl_introns_df_list.append(sig_diff_spl_introns_sig_group)

sig_diff_spl_introns = pd.concat(sig_diff_spl_introns_df_list, ignore_index=True)


# @markdown **Marker introns:** Select threshold for marker introns (> 0.0 means corresponding exon is included)
delta_psi_thres = 0.05 # @param {type:"number"}
marker_introns = sig_diff_spl_introns[sig_diff_spl_introns["delta_psi"] >= delta_psi_thres] # NEW

from crecerelle.utils import rank_intron_groups_groups
#rank_intron_groups_groups(adata_TU_tissue, sig_diff_spl_intron_groups, leiden_res)
rank_intron_groups_groups(adata_TU_tissue, sig_diff_spl_intron_groups, "leiden")

from crecerelle.utils import rank_introns_groups
#rank_introns_groups(adata_TU_tissue, sig_diff_spl_introns, leiden_res, "delta_psi", groups_test)
#rank_introns_groups(adata_TU_tissue, sig_diff_spl_introns, "leiden", "delta_psi", groups_test)
rank_introns_groups(adata_TU_tissue, marker_introns, "leiden", "delta_psi", sig_spl_groups) # NEW

In [ ]:
marker_introns

In [ ]:
adata_TU_tissue.uns["rank_intron_groups_groups"]

In [ ]:
adata_TU_tissue.uns["rank_introns_groups"]

In [ ]:
# @markdown **Number of unique significant isoforms:** Overall number of unique significantly spliced isoforms (Delta psi positive and negative)
sig_diff_spl_intron_groups_in_and_ex = diff_spl_intron_groups.query(f"p_value_adj < {p_value_adj_thres}")
sig_diff_spl_introns_in_and_ex = diff_spl_introns[diff_spl_introns["intron_group"].isin(sig_diff_spl_intron_groups_in_and_ex["name"].to_list())]
print(f"Number of significantly spliced isoforms (exon included and excluded) in {tax_level}: {len(sig_diff_spl_introns_in_and_ex["name"].unique())}")

In [ ]:
from crecerelle.plotting_utils import plot_customized_UMAP_coordinates
from matplotlib import gridspec
import numpy as np

# Calculate raw PSI scores
adata_TU_tissue.layers["PSI_raw"] = calculate_PSI(adata_TU_tissue)

# @markdown **Save figure:** Select if figure is saved
save_fig = False # @param {type:"boolean"}

# @markdown **Clustering group:** Select a clustering group for analysis
cluster_group = "0" # @param ["0", "1", "2", "3", "4", "5", "6", "7"]

# Set X_umap according to likelihood_key
adata_TU_tissue.obsm["X_umap"] = adata_TU_tissue.obsm[embedding_key + "_X_umap"].copy()

rename_isoforms = True

# Plot DSG analysis results
from crecerelle.plotting_utils import plot_dsg_analysis
plot_dsg_analysis(
    adata=adata_TU_tissue,
    cluster_group=cluster_group,
    cluster_groups=groups_test,
    model_name=model_name,
    taxonomy_level=tax_level,
    rename_isoforms=rename_isoforms,
    save_fig=save_fig,
    dataset_name=settings["dataset_name"],
    cell_type_key=settings["cell_type_key"]
)

In [ ]:
# Save DSGs of group_id as .csv
from crecerelle.utils import rank_introns_groups_df

# @markdown Save adata_TU_tissue and rank_introns_groups_df as csv
save_adata_TU_tissue = False # @param {type:"boolean"}
if save_adata_TU_tissue:
    adata_TU_tissue.write(path2data + "adata_TU_" + str(settings["num_hvg"]) + "_" + tax_level + "_trvi_inference.h5ad")

    rank_introns_groups_df = rank_introns_groups_df(adata_TU_tissue, cluster_group)
    rank_introns_groups_df.to_csv(path2data + "dsg_analysis_TRVI_" + embedding_key + "_" + tax_level + "_clustergroup_" + cluster_group + ".csv")

    print("AnnData file and csv file containing the differential splicing results have been saved")


## Combining gene expression and splicing to understand cell development

In [ ]:
# @markdown **Random seed:** select the random seed
seed = 8 # @param {type:"integer"} 0, 1, 2, 3, 4, 5, 6, 7, 8, 9
torch.manual_seed(seed)
np.random.seed(seed)

# @markdown **Taxonomy level:** Select the taxonomy level (e.g. tissue (i.e. Aorta, BAT, Bladder, Brain_Myeloid, Brain_Non-Myeloid etc.) in Tabula Muris) you want to analyse. If the dataset contains only one taxonomy level select None.
tax_level = "Brain_Non-Myeloid" # @param ["Aorta", "BAT", "Bladder", "Brain_Myeloid", "Brain_Non-Myeloid", "Diaphragm", "GAT", "Heart", "Kidney", "Large_Intestine", "Limb_Muscle", "Liver", "Lung", "MAT", "Mammary_Gland", "Marrow", "None","Pancreas", "SCAT", "Skin", "Spleen", "Thymus", "Tongue", "Trachea"]

# Load the inferred adata for specified tissue
adata_GE_tax_level = ad.read_h5ad(path2data + "adata_GE_" + str(settings["num_hvg"]) +"_" + tax_level + "_trvi_inference.h5ad")
adata_TU_tax_level = ad.read_h5ad(path2data + "adata_TU_" + str(settings["num_hvg"]) +"_" + tax_level + "_trvi_inference.h5ad")

print(f"The AnnData objects (gene expression and transcript usage) containing the inferences for taxonomy level {tax_level} are loaded")

# @markdown **Observation models of TRVI:** Select the observation model for gene expression as likelihood_key_1 (Gaussian, NB, ZINB) and for transcript usage as likelihood_key_2 (DM, ZANIDM, ZIDM).

likelihood_key_1 = "ZINB" # @param ["Gaussian", "NB", "ZINB"]
likelihood_key_2 = "ZIDM" # @param ["DM", "ZANIDM", "ZIDM"]

# @markdown **Cell embeddings:** Define the cell embeddings you want to analyse (e.g. S-GE, S-TU, GE-TU, P-GE, P-TU)
cell_embeddings_selected = ["S-GE", "S-TU", "GE-TU"] # @param

cell_embeddings_keys = []

if "S-GE" in cell_embeddings_selected:
    cell_embeddings_keys.append(likelihood_key_1 + "_shared_latent_mean")
if "S-TU" in cell_embeddings_selected:
    cell_embeddings_keys.append(likelihood_key_2 + "_shared_latent_mean")
if "GE-TU" in cell_embeddings_selected:
    cell_embeddings_keys.append(likelihood_key_1 + "_" + likelihood_key_2 + "_shared_latent_mean")
if "P-GE" in cell_embeddings_selected:
    cell_embeddings_keys.append(likelihood_key_1 + "_private_latent_mean")
if "P-TU" in cell_embeddings_selected:
    cell_embeddings_keys.append(likelihood_key_2 + "_private_latent_mean")

To analyse the differences of the individual cell embeddings inferred by TRVI, the toplogy of the latent space of cell embeddings is analysed. Therefore, the UMAP coordinates of the cell embeddings are calculated and the distance matrix visualised as heatmap gives insights in the topology of the latent space in terms of clustering.

In [ ]:
# @markdown **Topology of latent space of cell embeddings**

# @markdown **UMAP of cell emebeddings:** The selected cell embeddings are visualised as UMAPs. For the UMAPs, choose the umap_color_display_key (e.g. settings["cell_type_key"] by default, or "cell_ontology_class" for Tabula Muris)

umap_color_display_key = settings["cell_type_key"] # @param

# @markdown **Heatmap of distance matrices:** To visualise the topology of the latent space of the selected cell embeddings, the distance matrices are visualised as heatmaps. Choose to average each selected cell embeddings or if not choose a downsampling fraction to ease the computational load of calculating a full heatmap.
average_embeddings = False # @param {type:"boolean"}

adata_GE_tax_level_heatmap = adata_GE_tax_level.copy()
adata_TU_tax_level_heatmap = adata_TU_tax_level.copy()

downsampling_fraction = 0.1 # @param
if average_embeddings == False:
    # 1. Define your downsampling fraction (10% = 0.1)
    fraction = 0.1

    # 2. Use pandas to sample indices stratified by cell type
    # We group by the cell type column and sample 10% from each group (the same sample indices are applied to gene expression and transcript usage data to be consistent)
    sampled_indices = adata_GE_tax_level.obs.groupby(settings["cell_type_key"], group_keys=False).apply(
        lambda x: x.sample(frac=fraction, random_state=6)
    ).index

    # 3. Subset the AnnData object using the sampled indices
    adata_GE_tax_level_downsampled = adata_GE_tax_level[sampled_indices].copy()
    adata_TU_tax_level_downsampled = adata_TU_tax_level[sampled_indices].copy()

    adata_GE_tax_level_heatmap = adata_GE_tax_level_downsampled
    adata_TU_tax_level_heatmap = adata_TU_tax_level_downsampled

    # 4. Verify the results
    print(f"Original size of gene expression matrix: {adata_GE_tax_level.n_obs}")
    print(f"Downsampled size of gene expression matrix: {adata_GE_tax_level_heatmap.n_obs}")
    print(f"Original size of transcript usage matrix: {adata_TU_tax_level.n_obs}")
    print(f"Downsampled size of transcript usage matrix: {adata_TU_tax_level_heatmap.n_obs}")



# Compute distance matrix for each cell embedding
from crecerelle.utils import distance_matrix
list_of_distance_matrices = []
list_of_cluster_annotations = []
transcriptomic_facet_keys = []

for i, cell_embedding_selected in enumerate(cell_embeddings_selected):
    if cell_embedding_selected == "S-GE" or cell_embedding_selected == "GE-TU" or cell_embedding_selected == "P-GE":

        emb_distance_matrix, cluster_annotation = distance_matrix(
            adata_GE_tax_level_heatmap,
            cell_embeddings_keys[i],
            umap_color_display_key,
            average_embeddings=average_embeddings
        )
        list_of_distance_matrices.append(emb_distance_matrix)

        transcriptomic_facet_keys.append(cell_embedding_selected)

        print(f"For the cell embedding key {cell_embedding_selected} with cell embeddings {cell_embeddings_keys[i]} the distance matrix has been computed")

    elif cell_embedding_selected == "S-TU" or cell_embedding_selected == "P-TU":
        emb_distance_matrix, cluster_annotation = distance_matrix(
            adata_TU_tax_level_heatmap,
            cell_embeddings_keys[i],
            umap_color_display_key,
            average_embeddings=average_embeddings
        )
        list_of_distance_matrices.append(emb_distance_matrix)

        transcriptomic_facet_keys.append(cell_embedding_selected)

        print(f"For the cell embedding key {cell_embedding_selected} with cell embeddings {cell_embeddings_keys[i]} the distance matrix has been computed")
    else:
        raise ValueError("Invalid cell embedding selected")

# @markdown **Modality-relevance weights:** Select if the modality-relevance weights should be plotted as well
display_relevance_weights = True # @param {type:"boolean"}
if display_relevance_weights:
    rel_weight_key_1 = likelihood_key_1 + "_" + str(seed) + "_weighting"
    rel_weight_key_2 = likelihood_key_2 + "_" + str(seed) + "_weighting"
    rel_weight_key_3 = likelihood_key_1 + "_" + str(seed) + "_weighting"


    relevance_weight_keys = [rel_weight_key_1, rel_weight_key_2, rel_weight_key_3]
else:
    relevance_weight_keys = None
# @markdown **Save the figure panel:**

save_fig = False # @param {type:"boolean"}

from crecerelle.plotting_utils import plot_umap_and_distance_matrix
plot_umap_and_distance_matrix(
        adata_1 = adata_GE_tax_level,
        adata_2 = adata_TU_tax_level,
        cell_embeddings_keys = cell_embeddings_keys,
        transcriptomic_facet_keys = cell_embeddings_selected,
        relevance_weight_keys = relevance_weight_keys,
        distance_matrices =  list_of_distance_matrices,
        cluster_annotations = cluster_annotation,
        umap_color_display_key = umap_color_display_key,
        model_type = "trvi",
        save_fig = save_fig,
        random_state = seed,
        average_embeddings = average_embeddings,
        tax_level = tax_level,
        dataset_name = settings["dataset_name"]
)



Display the full latent space learnt by TRVI as a panel of UMAPs.

In [ ]:
from crecerelle.plotting_utils import plot_umap_latent_space_trvi

# @markdown **Display all latent spaces of TRVI:** The UMAPs of the latent spaces (P-GE, P-TU, S-GE, S-TU, GE-TU) are plotted colored according to color keys.
color_keys = [settings["cell_type_key"], settings["cell_type_key"]] # @param

# @markdown **Save figure:**
save_fig = False # @param {type:"boolean"}
file_suffix = "pdf" # @param ["pdf", "png", "svg"]

plot_umap_latent_space_trvi(
    adata_objects=(adata_GE_tax_level,adata_TU_tax_level),
    observation_model_keys=[likelihood_key_1, likelihood_key_2],
    color_keys=color_keys,
    seeds=[str(seed)],
    save_fig=save_fig,
    dataset_name=settings["dataset_name"],
    tax_level=tax_level,
    file_suffix=file_suffix,
    horizontal_legend_gap_mm=20.0
)

Plot the results of the combined DEG and DSG analysis on the chosen cell embedding (the function takes the last argument of cell_embeddings_selected i.e. GE-TU in default version)

In [ ]:
# @markdown **Results of the combined DEG and DSG analysis on the cell embeddings chosen**

sc.pp.neighbors(adata_GE_tax_level, use_rep=cell_embeddings_keys[-1], random_state=seed)
adata_GE_tax_level.obsm["X_umap"] = UMAP(n_components=2, random_state=seed).fit_transform(adata_GE_tax_level.obsm[cell_embeddings_keys[-1]])
adata_TU_tax_level.obsm["X_umap"] = UMAP(n_components=2, random_state=seed).fit_transform(adata_TU_tax_level.obsm[cell_embeddings_keys[-1]])

# @markdown **Save figure:** Tick the box to save the figure
save_fig = False # @param {type:"boolean"}

# Plot Leiden clusters and dotplots for supplementary material (Needs some adjustments)
from crecerelle.plotting_utils import plot_deg_and_dsg_analysis_trvi

plot_deg_and_dsg_analysis_trvi(
        adata=(adata_GE_tax_level, adata_TU_tax_level),
        tax_level=tax_level,
        model_name="TRVI_" + cell_embeddings_keys[-1],
        rename_isoforms=True,
        save_fig= save_fig,
        dataset_name=settings["dataset_name"],
        cell_type_key=settings["cell_type_key"],
        num_marker_genes=3,
        num_intron_group_markers=3
)

In [ ]:
# @markdown **Expression levels of top three DEGs:** The expression level of the top 3 DEGs is plotted for the specified cluster groups is plotted on the UMAP

# Plot top 3 DEGs per cluster for supplmentary material (might need some adjustments)
from crecerelle.plotting_utils import plot_differential_analysis_on_umap

# @markdown **Cluster groups:** Specify the Leiden cluster groups for which the top three DEGs are plotted
cluster_groups = ["0", "1", "2", "3", "4"] # @param

# @markdown **Number of DEGs plotted:** Specify the the number of DEGs to be displayed per cluster group
num_marker_genes = 3 # @param {type:"integer"}

# @markdown **Reference UMAP:** Choose which reference UMAP should be plotted (default is "leiden")
ref_umap_color = settings["cell_type_key"] # @param

# @markdown **Save figure:** Tick the box to save the figure
save_fig = False # @param {type:"boolean"}

plot_differential_analysis_on_umap(
        adata_GE_tax_level,
        cluster_groups,
        tax_level,
        "TRVI_" + cell_embeddings_keys[-1],
        "GE",
        renamed_isoforms = None,
        ref_umap_color=ref_umap_color,
        save_fig = save_fig,
        dataset_name=settings["dataset_name"],
        num_entities_per_group=num_marker_genes
)


In [ ]:
# @markdown **PSI score of top three DSG isoforms:** The PSI score of the top 3 DSG isoforms is plotted for the specified cluster groups is plotted on the UMAP

# Plot top 3 DSG isoforms per cluster for supplmentary material (might need some adjustments)
from crecerelle.plotting_utils import plot_differential_analysis_on_umap, rename_isoform_helper, dsg_dotplot_helper

# @markdown **Cluster groups:** Specify the Leiden cluster groups for which the top three DEGs are plotted
cluster_groups = ["0", "1", "2", "3", "4"] # @param

# @markdown **Number of isoform marker:** Specify the the number of isoform markers to be displayed per cluster group
num_isoform_markers = 3 # @param {type:"integer"}

# Rename isoforms
_, _, intron_names_dsg_plot = dsg_dotplot_helper(adata_TU_tax_level, cluster_groups)
renamed_isoforms = rename_isoform_helper(intron_names_dsg_plot)

# @markdown **Reference UMAP:** Choose which reference UMAP should be plotted (default is "leiden")
ref_umap_color = settings["cell_type_key"] # @param

# @markdown **Save figure:** Tick the box to save the figure
save_fig = False # @param {type:"boolean"}

plot_differential_analysis_on_umap(
        adata_TU_tax_level,
        cluster_groups,
        tax_level,
        "TRVI_" + cell_embeddings_keys[-1],
        "TU",
        renamed_isoforms = renamed_isoforms,
        ref_umap_color=ref_umap_color,
        save_fig = save_fig,
        dataset_name=settings["dataset_name"],
        num_entities_per_group=num_isoform_markers
)

In [ ]:
intron_names_dsg_plot

In [ ]:
# @markdown **Cell-type specific modality-relevance weights:** Display the mean of the gene expression and transcript usage relevance weight per cell type

cell_weights_1_mean = adata_GE_tax_level.obs.groupby(settings["cell_type_key"])[
        f"{likelihood_key_1}_weighting"].mean().round(3).reset_index()
cell_weights_2_mean = (
    adata_GE_tax_level.obs
    .groupby(settings["cell_type_key"], observed=False)[
        f"{likelihood_key_1}_weighting"
    ]
    .mean()
    .round(3)
    .rsub(1)
    .rename(f"{likelihood_key_2}_weighting")
    .reset_index()
)

cell_weights_1_mean.columns = [settings["cell_type_key"], "Weighting"]
cell_weights_1_mean["Modality"] = "GE"
cell_weights_2_mean.columns = [settings["cell_type_key"], "Weighting"]
cell_weights_2_mean["Modality"] = "TU"

weights_df = pd.concat([cell_weights_1_mean, cell_weights_2_mean], axis=0)
weights_df

For the selected taxonomy level and cell embeddings, the UMAPs are plotted together with a robustness analysis of the relevance weights. In addition, the dotplots of the top three DEGs and DSGs per Leiden cluster are plotted.

In [ ]:
# @markdown **Save figure:** Tick the box to save the figure panel
save_fig = False # @param {type:"boolean"}
file_suffix = "svg" # @param ["pdf", "png", "svg"]

# @markdown **Robustness across seeds:** Choose the seeds that shall be used for the robustness analysis of the relevance weights (e.g. [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
seeds_selected = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] # @param

# @markdown **Isoforms to visualise:** Name the two isoforms to visualise on the UMAP
# For brain non-myeloid isoform_1 = 'Gpm6a_chr8:54779651-55037327', isoform_2 = 'Gpm6a_chr8:54955316-55037327'
# Heart isoform_1 = 'Fbln2_chr6:91259771-91263338', isoform_2 = 'Fbln2_chr6:91257827-91263338'
isoform_1 = 'Gpm6a_chr8:54779651-55037327' # @param
isoform_2 = 'Gpm6a_chr8:54955316-55037327' # @param
isoforms_to_visualize = [isoform_1, isoform_2]

from crecerelle.plotting_utils import plot_cell_embeddings_analysis_trvi
plot_cell_embeddings_analysis_trvi(
        adata_objects = (adata_GE_tax_level, adata_TU_tax_level),
        cluster_groups = cluster_groups,
        likelihood_keys = [likelihood_key_1, likelihood_key_2],
        cell_embeddings_selected = cell_embeddings_selected,
        cell_embeddings_keys = cell_embeddings_keys,
        seeds_selected = seeds_selected,
        seed = seed,
        isoforms_to_visualize = isoforms_to_visualize,
        save_fig = save_fig,
        cell_type_key = settings["cell_type_key"],
        dataset_name = settings["dataset_name"],
        tax_level = tax_level,
        file_suffix = file_suffix
)

# Inference results cell atlas

In [ ]:
# @markdown **Save figure:**
save_fig = False # @param {type:"boolean"}

# @markdown **Embedding:** Select the embedding of TRVI
embedding_key = "ZINB_ZIDM_shared" # @param ["ZINB_shared", "ZINB_ZIDM_shared", "ZINB_private"]

embedding = embedding_key + "_latent_mean"

likelihood_key_1 = "ZINB"
likelihood_key_2 = "ZIDM"

likelihood_keys = [likelihood_key_1, likelihood_key_2]

shared_embedding_1 = "ZINB_shared_latent_mean"
shared_embedding_2 = "ZIDM_shared_latent_mean"
joint_embedding = "ZINB_ZIDM_shared_latent_mean"

# @markdown **Filter cells per taxonomy level:** Per taxonomy level (e.g. tissue), filter out all cell types with less than min_cells
min_cells = 50 # @param {type:"integer"}

#tissues = list(adata[0].obs[settings["cell_type_groups_key"]].unique())

if min_cells > 0:
    # 1. Calculate the size of each cell type group within each taxonomy level (tissue)
    # transform('size') returns a Series with the same index as the original dataframe
    counts = adata[0].obs.groupby([settings["cell_type_groups_key"], settings["cell_type_key"]])[settings["cell_type_key"]].transform('size')

    # 2. Create a boolean mask for cells to keep
    keep_mask = counts >= min_cells

    # 3. Slice both AnnData objects in the tuple
    adata_GE_filtered = adata[0][keep_mask, :].copy()
    adata_TU_filtered = adata[1][keep_mask, :].copy()

    adata_GE_filtered.obs[settings["cell_type_key"]] = adata_GE_filtered.obs[settings["cell_type_key"]].cat.remove_unused_categories()
    adata_TU_filtered.obs[settings["cell_type_key"]] = adata_TU_filtered.obs[settings["cell_type_key"]].cat.remove_unused_categories()

    adata = (adata_GE_filtered, adata_TU_filtered)


# Calculate UMAP for chosen embedding (add same X_umap to baoth)
adata[0].obsm["X_umap"] = UMAP(n_components=2, random_state=seed).fit_transform(adata[0].obsm[embedding])
adata[1].obsm["X_umap"] = adata[0].obsm["X_umap"].copy()

# @markdown **Functional enrichment analysis:** Specify up to 4 taxonomy level (tissue) members (e.g. Brain_Non-Myeloid, Heart, Marrow, GAT) for which functional enrichment on the set of DSGs and DEGs shall be carried out
tax_level_1 = "Brain_Non-Myeloid" # @param ["Aorta", "BAT","Bladder", "Brain_Myeloid","Brain_Non-Myeloid", "Diaphragm", "GAT", "Heart", "Large_Intestine", "Limb_Muscle", "Liver", "Lung", "Mammary_Gland","Marrow", "MAT", "Pancreas", "SCAT", "Skin", "Spleen", "Thymus", "Tongue", "Trachea"]
tax_level_2 = "Heart" # @param ["Aorta", "BAT","Bladder", "Brain_Myeloid","Brain_Non-Myeloid", "Diaphragm", "GAT", "Heart", "Large_Intestine", "Limb_Muscle", "Liver", "Lung", "Mammary_Gland","Marrow", "MAT", "Pancreas", "SCAT", "Skin", "Spleen", "Thymus", "Tongue", "Trachea"]
tax_level_3 = "Marrow" # @param ["Aorta", "BAT","Bladder", "Brain_Myeloid","Brain_Non-Myeloid", "Diaphragm", "GAT", "Heart", "Large_Intestine", "Limb_Muscle", "Liver", "Lung", "Mammary_Gland","Marrow", "MAT", "Pancreas", "SCAT", "Skin", "Spleen", "Thymus", "Tongue", "Trachea"]
tax_level_4 = "GAT" # @param ["Aorta", "BAT","Bladder", "Brain_Myeloid","Brain_Non-Myeloid", "Diaphragm", "GAT", "Heart", "Large_Intestine", "Limb_Muscle", "Liver", "Lung", "Mammary_Gland","Marrow", "MAT", "Pancreas", "SCAT", "Skin", "Spleen", "Thymus", "Tongue", "Trachea"]

tax_level_list = [tax_level_1, tax_level_2, tax_level_3, tax_level_4]

In [ ]:
# @markdown **Functional enrichment analysis:** The functional enrichment analysis is run for each specified taxonomy level based on their DEG and DSG sets. The results are saved
from crecerelle.utils import run_functional_enrichment_analysis, determine_shared_unique_deg_dsg_terms, load_merge_and_save_enrichment_terms

for tax_level_member in tax_level_list:
    print(f"Functional enrichment analysis for {tax_level_member}")
    # Run functional enrichment analysis for specified taxnonmy level member
    deg_enrichment_results_df = run_functional_enrichment_analysis(
        gene_types = "DEG",
        tissue = tax_level_member,
        num_hvg = settings["num_hvg"],
        path2data = "./data/" + settings["dataset_name"] + "/",
        organism = "mmusculus"
    )

    dsg_enrichment_results_df = run_functional_enrichment_analysis(
        gene_types = "DSG",
        tissue = tax_level_member,
        num_hvg = settings["num_hvg"],
        path2data = "./data/" + settings["dataset_name"] + "/",
        organism = "mmusculus"
    )

    # Determine shared and unique enriched terms and store them in dataframes
    dsg_unique_terms_df, deg_unique_terms_df, overlapping_terms_df = determine_shared_unique_deg_dsg_terms(
        deg_enrichment_results_df,
        dsg_enrichment_results_df,
        tax_level_member,
        True,
    )

    # Upsetplot of sets of enriched terms
    from crecerelle.plotting_utils import plot_shared_and_unique_enrichment_terms_upsetplot

    num_deg_terms = len(deg_unique_terms_df)
    num_dsg_terms = len(dsg_unique_terms_df)
    num_overlapping_terms = len(overlapping_terms_df)

    # Create upsetplot
    plot_shared_and_unique_enrichment_terms_upsetplot(
        num_deg_terms,
        num_dsg_terms,
        num_overlapping_terms,
        tax_level_member,
        settings["dataset_name"],
        True
    )

# Load and merge the terms of each tax level

# DSG-unique terms
enrichment_results_dsg_unique_df_all = load_merge_and_save_enrichment_terms(
    tax_level_list=tax_level_list,
    file_prefix="dsg_unique_terms_df",
    output_filename="dsg_unique_terms_df_all.csv",
    path2data=path2data,
    dataset_name=settings["dataset_name"]
)

# DEG-unique terms
enrichment_results_deg_unique_df_all = load_merge_and_save_enrichment_terms(
    tax_level_list=tax_level_list,
    file_prefix="deg_unique_terms_df",
    output_filename="deg_unique_terms_df_all.csv",
    path2data=path2data,
    dataset_name=settings["dataset_name"]
)

# Overlapping terms
enrichment_results_overlapping_df_all = load_merge_and_save_enrichment_terms(
    tax_level_list=tax_level_list,
    file_prefix="overlapping_terms_df",
    output_filename="overlapping_terms_df_all.csv",
    path2data=path2data,
    dataset_name=settings["dataset_name"]
)


In [ ]:
# This is necessary if you want to have the GO-Figure plot for the pathways
!pip install adjustText
!git clone https://gitlab.com/evogenlab/GO-Figure.git

In [ ]:
# Plotting
from matplotlib import gridspec
import math
import seaborn as sns
from crecerelle.plotting_utils import plot_customized_UMAP_coordinates

# @markdown **Cell state variations and DSG-associated biological pathways:** Analyze the variations of gene expression and transcript usage relevance weights across different cell types and cell type groups. Compare the DSG and DEG associated enrichment terms, plot DSG-associated and DEG-DSG-associated biological pathways

# @markdown **Save figure:** Tick the box to save the figure panel
save_fig = False # @param {type:"boolean"}
file_suffix = "pdf" # @param ["pdf", "png", "svg"]

from crecerelle.plotting_utils import plot_transcript_usage_cell_states_and_pathways_trvi
from crecerelle.cell import TABULA_MURIS_TISSUE_ORGAN_SYSTEM_DICT

plot_transcript_usage_cell_states_and_pathways_trvi(
        adata_objects = adata,
        enrichment_results_dsg_unique_df_all = enrichment_results_dsg_unique_df_all,
        enrichment_results_overlapping_df_all = enrichment_results_overlapping_df_all,
        tax_level_list = tax_level_list,
        embedding = embedding,
        seed = seed,
        cell_org_hierarchy_dictionary = TABULA_MURIS_TISSUE_ORGAN_SYSTEM_DICT,
        save_fig = save_fig,
        cell_type_groups_key = settings["cell_type_groups_key"],
        cell_type_key = settings["cell_type_key"],
        dataset_name = settings["dataset_name"],
        file_suffix = file_suffix
)

In [ ]:
# @markdown **Functional enrichment results:** Load functional enrichment results dataframe for a specific tissue
tissue = "Brain_Non-Myeloid" # @param ["Aorta", "BAT","Bladder", "Brain_Myeloid","Brain_Non-Myeloid", "Diaphragm", "GAT", "Heart", "Large_Intestine", "Limb_Muscle", "Liver", "Lung", "Mammary_Gland","Marrow", "MAT", "Pancreas", "SCAT", "Skin", "Spleen", "Thymus", "Tongue", "Trachea"]
enriched_terms_set = "DSG" # @param ["DEG", "DSG", "both"]

if enriched_terms_set == "DEG":
    enrichment_results_df = pd.read_csv("./data/" + settings["dataset_name"] + "/deg_unique_terms_df_" + tissue + ".csv")
elif enriched_terms_set == "DSG":
    enrichment_results_df = pd.read_csv("./data/" + settings["dataset_name"] + "/dsg_unique_terms_df_" + tissue + ".csv")
elif enriched_terms_set == "both":
    enrichment_results_df = pd.read_csv("./data/" + settings["dataset_name"] + "/overlapping_terms_df_" + tissue + ".csv")
else:
    raise ValueError("Invalid pathway_set value. Must be 'DEG', 'DSG', or 'both'.")

from crecerelle.plotting_utils import plot_enrichment_term_barchart
fig, _ = plot_enrichment_term_barchart(
            enrichment_results_df,
            title="DSG-unique biological pathways",
            top_n=40,
            sources="GO:BP",  # None = all sources; e.g. "GO:BP" = GO biological process only
            order_by="dataframe",
            show_rank=False,
            pvalue_scale="log",  # use "linear" for a linear p-value color scale
            pvalue_label="adjusted p-value",
            plot_term_id = True,
            p_max = 0.05
)

enrichment_results_df.head(40)



In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# @markdown **Functional enrichment analysis results:** Specify up to 4 taxononmy levels (e.g. tissues) for which the specified functional enrichment terms determined on a set of DEGs and DSGs shall be compared.

tax_level_1 = 'Brain_Non-Myeloid' # @param ["Aorta", "BAT","Bladder", "Brain_Myeloid","Brain_Non-Myeloid", "Diaphragm", "GAT", "Heart", "Large_Intestine", "Limb_Muscle", "Liver", "Lung", "Mammary_Gland","Marrow", "MAT", "Pancreas", "SCAT", "Skin", "Spleen", "Thymus", "Tongue", "Trachea"]
tax_level_2 = 'Heart' # @param ["Aorta", "BAT","Bladder", "Brain_Myeloid","Brain_Non-Myeloid", "Diaphragm", "GAT", "Heart", "Large_Intestine", "Limb_Muscle", "Liver", "Lung", "Mammary_Gland","Marrow", "MAT", "Pancreas", "SCAT", "Skin", "Spleen", "Thymus", "Tongue", "Trachea"]
tax_level_3 = 'Marrow' # @param ["Aorta", "BAT","Bladder", "Brain_Myeloid","Brain_Non-Myeloid", "Diaphragm", "GAT", "Heart", "Large_Intestine", "Limb_Muscle", "Liver", "Lung", "Mammary_Gland","Marrow", "MAT", "Pancreas", "SCAT", "Skin", "Spleen", "Thymus", "Tongue", "Trachea"]
tax_level_4 = 'GAT' # @param ["Aorta", "BAT","Bladder", "Brain_Myeloid","Brain_Non-Myeloid", "Diaphragm", "GAT", "Heart", "Large_Intestine", "Limb_Muscle", "Liver", "Lung", "Mammary_Gland","Marrow", "MAT", "Pancreas", "SCAT", "Skin", "Spleen", "Thymus", "Tongue", "Trachea"]

tax_level_list = [tax_level_1, tax_level_2, tax_level_3, tax_level_4]

# @markdown **Enrichment term:** Specify the enrichment terms to be filtered for (e.g. "GO:BP", "HP"). if None is given all will be considered
sources = None # @param

# @markdown **Save figure:** Tick the box to save the figure panel
save_fig = False # @param {type:"boolean"}
file_suffix = "pdf" # @param ["pdf", "png", "svg"]

from crecerelle.plotting_utils import plot_enrichment_term_set_comparison
plot_enrichment_term_set_comparison(
        dataset_name=settings["dataset_name"],
        tax_level_list=tax_level_list,
        sources=sources,
        save_fig=save_fig,
        file_suffix=file_suffix,
)

In [ ]:
from crecerelle.plotting_utils import plot_functional_enrichment_bubble

fig, ax = plt.subplots(figsize=(10, 8))

# @markdown **Biological pathway enrichment bubble plot with adjusted p-value**

# @markdown **Choose the enrichment resuls:** Select the enrichment results for DSG, DEG or both DSG and DEG associated biological pathways
enriched_terms_set = 'DSG' # @param ['DEG', 'DSG', 'DEG and DSG']

if enriched_terms_set == "DEG":
    enrichment_results_df_selected = pd.read_csv("./data/" + settings["dataset_name"] + "/deg_unique_terms_df_all.csv")
    plot_title = "DEG-unique functional enrichment analysis"
elif enriched_terms_set == "DSG":
    enrichment_results_df_selected = pd.read_csv("./data/" + settings["dataset_name"] + "/dsg_unique_terms_df_all.csv")
    plot_title = "DSG-unique functional enrichment analysis"
elif enriched_terms_set == "DEG and DSG":
    enrichment_results_df_selected = pd.read_csv("./data/" + settings["dataset_name"] + "/overlapping_terms_df_all.csv")
    plot_title = "DEG and DSG-unique functional enrichment analysis"
else:
    raise ValueError("Invalid pathway_set value. Must be 'DEG', 'DSG', or 'DEG and DSG'.")

result = plot_functional_enrichment_bubble(
    enrichment_results_df_selected,
    ax=ax,
    gofigure_path="./GO-Figure",
    source="GO:BP",
    title=plot_title,
)

fig.subplots_adjust(bottom=0.40)

# @markdown **Save figure:**
save_fig = False # @param {type:"boolean"}
file_suffix = "pdf" # @param ["pdf", "png", "svg"]

if save_fig:
    fig_name  = "./figures/" + settings["dataset_name"] + "/" + enriched_terms_set + "_functional_enrichment_bubble_plot_adj_p_value." + file_suffix
    fig.savefig(
        fig_name,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.3,
        bbox_extra_artists=result.extra_artists,
    )

plt.show()